> **Note:** this notebook is the original classroom EDA and merges tables in memory, so it works on a **1M-row sample** of the prior file (`prior_rows=1_000_000`). Full-dataset analytics, market basket analysis and the ML model live in the Streamlit app (`streamlit run app.py`), which is built on chunked aggregates instead of a 32M-row merge.


# CartIQ — Instacart Market Basket Analysis
### Group 13

**Members:**
- MOHNISH RAJ YADAV — 2500030928
- DARIPALLY BILVANTH RAM — 2500031922

**Objective:**
1. Analyze basket/order behavior
2. Analyze product reorder behavior

**Scope of this notebook:** Data Loading → Data Understanding → Data Cleaning →
Data Integration → EDA → EDA Insights.

No machine learning model is trained in this notebook — that is a later phase.

## 1. Import Libraries

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_datasets
from src.data_cleaning import (
    explore_dataframe, clean_orders, clean_products,
    clean_order_products, clean_aisles_departments
)
from src.eda import (
    build_merged_data, univariate_analysis, product_analysis,
    basket_analysis, reorder_analysis, time_based_analysis,
    customer_behavior_analysis, correlation_analysis, print_eda_insights
)

sns.set_style("whitegrid")
%matplotlib inline

## 2. Load Dataset

Make sure the Instacart CSV files are placed inside the `data/` folder (see `data/README.md`).

In [ ]:
dataframes = load_datasets(prior_rows=1_000_000)  # SAMPLE of the 32M-row file

orders = dataframes.get("orders")
products = dataframes.get("products")
order_products_prior = dataframes.get("order_products_prior")
aisles = dataframes.get("aisles")
departments = dataframes.get("departments")

## 3. Dataset Overview

A quick look at each table before deeper exploration.

In [ ]:
for name, df in dataframes.items():
    print(name, "->", df.shape)

## 4. Data Understanding

Each table represents a different part of the Instacart dataset:

- **orders** → order-level information (customer, order number, day, hour, days since prior order)
- **products** → product information (name, aisle, department)
- **aisles** → aisle/category information
- **departments** → department information
- **order_products_prior** → which products were in each historical order, and whether it was a reorder

In [ ]:
explore_dataframe(orders, "orders")

In [ ]:
explore_dataframe(products, "products")

In [ ]:
explore_dataframe(order_products_prior, "order_products_prior")

In [ ]:
explore_dataframe(aisles, "aisles")
explore_dataframe(departments, "departments")

## 5. Data Cleaning

Basic, explainable cleaning: removing duplicates, fixing data types, and
noting expected missing values (e.g. `days_since_prior_order` is naturally
missing for a customer's first order).

In [ ]:
orders = clean_orders(orders)
products = clean_products(products)
order_products_prior = clean_order_products(order_products_prior)
aisles, departments = clean_aisles_departments(aisles, departments)

## 6. Data Integration

We merge `order_products_prior` with `orders`, `products`, `aisles`, and
`departments` to build one combined dataset (`merged_data`) suitable for EDA.

In [ ]:
merged_data = build_merged_data(orders, order_products_prior, products, aisles, departments)
merged_data.head()

## 7. Univariate Analysis

Individual variable distributions: orders per customer, products per order,
order day/hour, and days since prior order.

In [ ]:
univariate_analysis(orders, merged_data)

## 8. Product Analysis

Which products, departments, and aisles are most popular, and which
products are reordered most often.

In [ ]:
product_results = product_analysis(merged_data)

## 9. Basket Analysis

How many products customers typically buy per order.

In [ ]:
basket_stats, basket_sizes = basket_analysis(merged_data)

## 10. Reorder Analysis

The `reordered` column is the existing target/label that a future ML phase
could predict. Here we only explore it descriptively — no model is trained.

In [ ]:
reorder_results = reorder_analysis(merged_data)

## 11. Time-Based Analysis

When orders happen: busiest day and hour, and a day/hour heatmap.

In [ ]:
time_results = time_based_analysis(orders)

### Customer / Order Behavior

Average products per order, and how reorder rate changes with order number.

In [ ]:
customer_results = customer_behavior_analysis(orders, merged_data)

## 12. Correlation Analysis

Correlation is calculated only for meaningful numerical variables (not ID
columns). Remember: **correlation does not imply causation.**

In [ ]:
corr_matrix = correlation_analysis(orders, merged_data)
corr_matrix

## 13. EDA Insights

A summary of findings, computed directly from the real dataset.

In [ ]:
print_eda_insights(product_results, basket_stats, reorder_results, time_results)

## 14. Conclusion

This notebook completed the data preparation and exploratory analysis phase
of CartIQ: loading, understanding, cleaning, integrating, and exploring the
Instacart dataset. Key patterns in basket size, product popularity, and
reorder behavior were identified.

**Next steps (not part of this deliverable):** building and evaluating a
machine learning model to predict reorder behavior.